<a href="https://colab.research.google.com/github/saifrx7/saif-ww/blob/main/Data_preprocessing_and_Cleaning_Steps_using_sample_Dataset_BCA3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#For BCA 3 Lab:                                                                                                                                    Problem: You have been given a dataset in Excel format, for an example, a sample dataset in the healthcare domain. First, you need to convert it to CSV format and perform the following tasks in the Python Programming Language:

# Problem 1: Working in a Jupyter/IDE in a Python environment, using NumPy and Pandas to import CSV datasets and perform basic data operations.

# Problem 2: Handling missing values; null values; duplicate values; data cleaning; feature selection; train—test split; visualization using Matplotlib and EDA operations on the same dataset.


**Please Note: Dataset may be any Dataset as per your choice and according to your requirement just for an example here we choose Sample Dataset (stroke dataset) from medical/healthcare domain. You can choose your own domain but basic Data Preprocessing and cleaning operation will be remaining same. As and when required, you need to change your Colab Python code as per your requirement and functions/operations you want to do.**



# Step by Step Procedure to Solve This problem:

# Sample Dataset — EDA, Preprocessing & Feature Selection


**Run this notebook top-to-bottom in Google Colab.**

Pipeline covered:
1. Install & import libraries
2. Upload the dataset
3. Initial exploration
4. Handle missing / null values
5. Handle duplicate values
6. Exploratory Data Analysis (EDA) & visualization
7. Data normalization (numerical scaling + categorical encoding)
8. Correlation analysis
9. Feature selection & feature ranking
10. Train-test split (80:20)




## 1. Install & import libraries
Colab already ships with these, but running this once ensures the right versions are present.

In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn openpyxl --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.1.3
Pandas version: 2.2.3


## 2. Upload the dataset

Run the cell below, then click **Choose Files** and select `sample_dataset.xlsx`
from your computer. (If your file already lives in Google Drive instead, use the
commented-out `drive.mount(...)` block below it.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # select stroke_sample_dataset.xlsx when prompted

filename = list(uploaded.keys())[0]
print("Uploaded:", filename)

# --- Alternative: load from Google Drive instead of uploading each time ---
# from google.colab import drive
# drive.mount('/content/drive')
# filename = "/content/drive/MyDrive/stroke_sample_dataset.xlsx"


In [ ]:
df = pd.read_excel(filename)
print("Shape (rows, columns):", df.shape)
df.head()

## 3. Initial data exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df['stroke'].value_counts())
print(df['stroke'].value_counts(normalize=True) * 100)

> The target is heavily imbalanced — only ~4.9% of records are positive stroke cases. Keep this in mind for later modeling (e.g. `class_weight='balanced'`, stratified splitting).

## 4. Handling missing / null values

In [ ]:
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Missing Value Heatmap (Before Cleaning)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
missing_before[missing_before > 0].sort_values(ascending=False).plot(kind="bar", color="salmon")
plt.title("Count of Missing Values per Column (Before Cleaning)")
plt.ylabel("Number of missing values")
plt.show()

**Imputation strategy**
- `bmi` (numerical) → filled with the column **median** (robust to outliers)


In [ ]:
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df["smoking_status"] = df["smoking_status"].fillna("Unknown")

print(df.isnull().sum().sum(), "missing values remain")

## 5. Handling duplicate values

In [ ]:
dup_count = df.duplicated().sum()
print("Duplicate rows found:", dup_count)

df = df.drop_duplicates()
df = df.drop(columns=["id"])   # id is a unique identifier, not predictive
print("Shape after cleaning:", df.shape)

## 6. Exploratory Data Analysis (EDA) & visualization

In [ ]:
numerical_cols = ["age", "avg_glucose_level", "bmi"]
categorical_cols = ["gender", "hypertension", "heart_disease", "ever_married",
                     "work_type", "Residence_type", "smoking_status"]

### 6.1 Distribution of numerical features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, numerical_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

### 6.2 Boxplots — outlier detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, numerical_cols):
    sns.boxplot(y=df[col], ax=ax, color="lightgreen")
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

### 6.3 Categorical feature counts

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()
for ax, col in zip(axes, categorical_cols + ["stroke"]):
    sns.countplot(x=df[col], ax=ax, hue=df[col], palette="Set2", legend=False)
    ax.set_title(f"Count of {col}")
    ax.tick_params(axis="x", rotation=30)
for ax in axes[len(categorical_cols) + 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6.4 Target class balance

In [ ]:
plt.figure(figsize=(5, 5))
df["stroke"].value_counts().plot.pie(
    labels=["No Stroke", "Stroke"], autopct="%1.1f%%",
    colors=["#66b3ff", "#ff6666"], startangle=90
)
plt.title("Stroke vs No-Stroke Proportion")
plt.ylabel("")
plt.show()

### 6.5 Numerical features vs. target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(x="stroke", y="age", data=df, hue="stroke", palette="Set1", legend=False, ax=axes[0])
axes[0].set_title("Age by Stroke Outcome")
sns.boxplot(x="stroke", y="avg_glucose_level", data=df, hue="stroke", palette="Set1", legend=False, ax=axes[1])
axes[1].set_title("Avg Glucose Level by Stroke Outcome")
plt.tight_layout()
plt.show()

## 7. Data normalization (numerical & categorical)

- **Categorical → numeric:** Label Encoding
- **Numerical → common scale:** StandardScaler (z-score); Min-Max is also shown for comparison

### 7.1 Encode categorical columns

In [ ]:
df_processed = df.copy()

label_enc_cols = ["gender", "ever_married", "Residence_type", "work_type", "smoking_status"]
encoders = {}
for col in label_enc_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"Encoded '{col}': {encoders[col]}")

### 7.2 Scale numerical columns

In [ ]:
scaler = StandardScaler()
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])
print(df_processed[numerical_cols].describe().loc[["mean", "std"]])

In [ ]:
# Min-Max scaled version, shown only for visual comparison
minmax = MinMaxScaler()
df_minmax_demo = pd.DataFrame(minmax.fit_transform(df[numerical_cols]), columns=numerical_cols)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df_processed["age"], kde=True, ax=axes[0], color="purple")
axes[0].set_title("Age after Standard Scaling (z-score)")
sns.histplot(df_minmax_demo["age"], kde=True, ax=axes[1], color="orange")
axes[1].set_title("Age after Min-Max Scaling (0-1)")
plt.tight_layout()
plt.show()

## 8. Correlation analysis

In [ ]:
corr = df_processed.corr()
print(corr["stroke"].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Correlation Heatmap (All Features incl. Target)")
plt.show()

## 9. Feature selection & feature ranking

Two complementary methods:
- **ANOVA F-test** (`SelectKBest`) — statistical, univariate
- **Random Forest importance** — captures non-linear relationships & interactions

### 9.1 ANOVA F-test

In [ ]:
X = df_processed.drop(columns=["stroke"])
y = df_processed["stroke"]

selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X, y)
anova_scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(anova_scores)

In [ ]:
plt.figure(figsize=(9, 5.5))
anova_scores.plot(kind="barh", color="teal")
plt.gca().invert_yaxis()
plt.title("Feature Ranking - ANOVA F-test Scores")
plt.xlabel("F-score")
plt.show()

### 9.2 Random Forest feature importance

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X, y)
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(rf_importance)

In [ ]:
plt.figure(figsize=(9, 5.5))
rf_importance.plot(kind="barh", color="darkorange")
plt.gca().invert_yaxis()
plt.title("Feature Ranking - Random Forest Importance")
plt.xlabel("Importance")
plt.show()

top_features = rf_importance.head(6).index.tolist()
print("Top 6 selected features:", top_features)

## 10. Train-test split (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("\nTrain target distribution:\n", y_train.value_counts(normalize=True) * 100)
print("\nTest target distribution:\n", y_test.value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(["Train (80%)", "Test (20%)"], [len(X_train), len(X_test)], color=["#4c72b0", "#dd8452"])
plt.title("Train-Test Split Sizes")
plt.ylabel("Number of samples")
for i, v in enumerate([len(X_train), len(X_test)]):
    plt.text(i, v + 200, str(v), ha="center")
plt.show()

## 11. Save the processed dataset

Downloads `stroke_processed.csv` to your computer — ready to plug straight into a
classification model (Logistic Regression, Random Forest, XGBoost, etc.).

In [ ]:
df_processed.to_csv("stroke_processed.csv", index=False)

from google.colab import files
files.download("stroke_processed.csv")

# Note: Now We will apply Machine Learning Classifiers as per our application requirement on Processed Data. So in this Example we have complete our data preprocessing and cleaning steps by using sample stroke dataset.